# 04 — Node & Behavioral Feature Engineering

## Objective
Build leakage-safe transaction, temporal, entity-history, and shared-infrastructure features using only information available before the validation/test period.

> **Scientific contract:** fraud labels are validation ground truth, never unsupervised predictors. Chronological splits protect against future leakage. The test period is locked until Notebook 11.

### Outputs
Reproducible artifacts are saved to `artifacts/` and audit evidence to `reports/`. Interactive controls are for investigation/exploration; the underlying tables remain reproducible.

## 1. Fit history on train only

In [1]:
from pathlib import Path
import sys, json, warnings, numpy as np, pandas as pd
import plotly.express as px
from IPython.display import display, Markdown
import ipywidgets as widgets
warnings.filterwarnings("ignore")
ROOT=Path.cwd()
if not (ROOT/"data").exists(): ROOT=ROOT.parent
sys.path.insert(0,str(ROOT/"src"))
from pipeline_utils import *
ART=ROOT/"artifacts"; REP=ROOT/"reports"; ART.mkdir(exist_ok=True); REP.mkdir(exist_ok=True)
RANDOM_STATE=42
fraud,_=load_data(); train,val,test,bounds=chronological_split(fraud); builder=HistoryFeatureBuilder().fit(train); Xtr,Xv,Xt=map(builder.transform,[train,val,test]); cols=feature_columns(Xtr); display(Markdown(f"**Feature count:** {len(cols)}")); display(Xtr[cols].describe().T)

**Feature count:** 15

,count,mean,std,min,25%,50%,75%,max
purchase_value,105778.0,36.878907,18.329275,9.000000,22.000000,34.000000,49.000000,154.000000
value_log1p,105778.0,3.508378,0.520925,2.302585,3.135494,3.555348,3.912023,5.043425
age,105778.0,33.159476,8.622335,18.000000,27.000000,33.000000,39.000000,76.000000
account_age_hours,105778.0,1170.502048,846.100142,0.000278,423.379931,1077.910139,1859.151528,2879.992222
purchase_hour,105778.0,11.510248,6.911520,0.000000,6.000000,12.000000,17.000000,23.000000
purchase_dow,105778.0,3.007828,2.014491,0.000000,1.000000,3.000000,5.000000,6.000000
is_weekend,105778.0,0.290193,0.453853,0.000000,0.000000,0.000000,1.000000,1.000000
user_id_history_count,105778.0,1.000000,0.000000,1.000000,1.000000,1.000000,1.000000,1.000000
device_id_history_count,105778.0,1.919832,3.081064,1.000000,1.000000,1.000000,1.000000,20.000000
ip_address_history_count,105778.0,1.861030,3.067063,1.000000,1.000000,1.000000,1.000000,20.000000


## 2. Feature catalog

In [2]:
family=lambda c:"transaction" if "value" in c else ("temporal" if c in ["age","account_age_hours","purchase_hour","purchase_dow","is_weekend"] else "graph_behavior"); catalog=pd.DataFrame({"feature":cols,"family":[family(c) for c in cols]}); display(catalog)

,feature,family
0,purchase_value,transaction
1,value_log1p,transaction
2,age,temporal
3,account_age_hours,temporal
4,purchase_hour,temporal
5,purchase_dow,temporal
6,is_weekend,temporal
7,user_id_history_count,graph_behavior
8,device_id_history_count,graph_behavior
9,ip_address_history_count,graph_behavior


## 3. Interactive feature explorer

In [3]:
sel=widgets.SelectMultiple(options=cols,value=tuple(cols[:8]),description="Features"); n=widgets.IntSlider(value=20,min=5,max=50,description="Rows"); out=widgets.Output()
def show(*_):
    with out: out.clear_output(); display(Xtr.loc[:,list(sel.value)].head(n.value))
sel.observe(show,'value'); n.observe(show,'value'); display(widgets.VBox([sel,n,out])); show()
Xtr[cols].to_parquet(ART/"train_features.parquet"); Xv[cols].to_parquet(ART/"validation_features.parquet"); Xt[cols].to_parquet(ART/"test_features.parquet"); save_json({"features":cols,"bounds":bounds},ART/"feature_config.json")